# XMCD Visualiser
### *{{beamline}}: {{date}}*
This notebook allows the user to interactively choose pairs of XAS measurements to subtract as XMCD measurements, using different backgrounds, then average these to generate a final XMCD pattern.

{{description}}

*mmg_toolbox* contains some useful tools for analysis of XAS spectra, in particular for calculation of subtracted
polarised spectra for calculation of dichroic signals like XMCD and XMLD.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, Latex, clear_output

import numpy as np
import matplotlib.pyplot as plt
from mmg_toolbox import Experiment, module_info, xas
from mmg_toolbox.plotting.matplotlib import set_plot_defaults

print(module_info())
set_plot_defaults()  # set nice defaults for matplotlib plots

## Load Spectra


In [ ]:
# Create experiment object - monitors one or more data folders for files
exp = Experiment('{{experiment_dir}}', instrument='{{beamline}}')

print(exp)

# print all scans in directory (could take a while...)
# print(exp.all_scans_str())

In [ ]:
# pick scan range
scan_range = {{scan_numbers}}
mode = 'tey'
sample_name=  'mysample'

# Load scan data and plot raw spectra
spectras = exp.load_xas(*scan_range, sample_name=sample_name, mode=mode)  # only loads NXxas spectra (energy scans) and creates a Spectra object

for xas_scan in spectras:
    print(xas_scan)


In [ ]:
# Choose Pairs (choose one of the options below to define the pairs)
# 1. Automatic choice
pol_pairs = xas.polarised_pairs(*spectras)

# 2. Manual choice
# scan_numbers = [
#     # (pol1, pol2)
#     (37436, 37438),
#     (37437, 37439),
#
# ]
# pol_pairs = [
#     exp.load_xas(s1, s2, sample_name=sample_name, mode=mode, match_metadata=False)
#     for s1, s2 in scan_numbers
# ]

# show table
s = f'| {pol_pairs[0][0].metadata.pol} | {pol_pairs[0][1].metadata.pol} |\n| --- | --- |\n'
for p1_scan, p2_scan in pol_pairs:
    s += f"| {p1_scan.metadata.scan_no} | {p2_scan.metadata.scan_no} |\n"
display(Markdown(s))

In [ ]:
def make_line_plot(scan1: xas.SpectraContainer, scan2: xas.SpectraContainer, background: str | None = None):
    scan1_proc = scan1.divide_by_preedge()
    scan2_proc = scan2.divide_by_preedge()
    if background is not None:
        scan1_proc = scan1_proc.remove_background(background)
        scan2_proc = scan2_proc.remove_background(background)
    subtract = scan1_proc - scan2_proc
    subtract.create_figure()



In [1]:

# --------------------------------------------------
# Create plot mapping (name -> function call)
# --------------------------------------------------

plots = {
    f"Plot {i}": lambda: make_line_plot(s1, s2)
    for i, (s1, s2) in enumerate(pol_pairs)
}

# --------------------------------------------------
# Create checkboxes
# --------------------------------------------------

checkboxes = {
    name: widgets.Checkbox(value=False, description=name)
    for name in plots
}

# --------------------------------------------------
# Output area
# --------------------------------------------------

output = widgets.Output()

# --------------------------------------------------
# Update function
# --------------------------------------------------

def update_plots(change=None):
    with output:
        clear_output()

        selected = [name for name, cb in checkboxes.items() if cb.value]

        if not selected:
            print("No plots selected")
            return

        # Create stacked subplots (one per selected plot)
        fig, axes = plt.subplots(len(selected), 1, figsize=(8, 3 * len(selected)))

        if len(selected) == 1:
            axes = [axes]

        for ax, name in zip(axes, selected):
            plt.sca(ax)
            plots[name]()
            ax.grid(True)

        plt.tight_layout()
        plt.show()

# --------------------------------------------------
# Link checkbox events
# --------------------------------------------------

for cb in checkboxes.values():
    cb.observe(update_plots, "value")

# --------------------------------------------------
# Select All / Clear All buttons
# --------------------------------------------------

def select_all(b):
    for cb in checkboxes.values():
        cb.value = True

def clear_all(b):
    for cb in checkboxes.values():
        cb.value = False

btn_all = widgets.Button(description="Select All")
btn_none = widgets.Button(description="Clear All")

btn_all.on_click(select_all)
btn_none.on_click(clear_all)

# --------------------------------------------------
# Accordion layout
# --------------------------------------------------

checkbox_list = widgets.VBox(list(checkboxes.values()))
accordion = widgets.Accordion(children=[checkbox_list])
accordion.set_title(0, "Select Plots")

# --------------------------------------------------
# Display UI
# --------------------------------------------------

controls = widgets.VBox([
    widgets.HBox([btn_all, btn_none]),
    accordion
])

display(controls, output)

# Initial render
update_plots()


NameError: name 'pol_pairs' is not defined

In [ ]:
# Plot each RAW spectra
n_spectra = len(spectras[0].spectra)

fig, axes = plt.subplots(1, n_spectra, figsize=[6 * n_spectra, 6], dpi=100, squeeze=False)
for scan in spectras:
    for ax, (mode, spectra) in zip(axes.flat, scan.spectra.items()):
        spectra.plot(ax)
        ax.set_ylabel(f"{mode} / monitor")
        ax.set_xlabel('E [eV]')
        ax.legend()

fig.tight_layout()

### Process the spectra

In [ ]:
spectras = [spectra.divide_by_preedge().remove_background('linear') for spectra in spectras]

fig, axes = plt.subplots(1, n_spectra, figsize=[6 * n_spectra, 6], dpi=100, squeeze=False)
for scan in spectras:
    for ax, (mode, spectra) in zip(axes.flat, scan.spectra.items()):
        spectra.plot(ax)
        ax.set_title(spectra.process_label)
        ax.set_ylabel(mode)
        ax.set_xlabel('E [eV]')
        ax.legend()

fig.tight_layout()

### Separate the spectra by polarisation
The *average_polarised_scans* function will average the spectra into opposite polarisations, based on the polarisation of the first spectra.

In [ ]:
# Average polarised scans
for xas_scan in spectras:
    print(f"{xas_scan.name}: {xas_scan.metadata.pol}")
pol1, pol2 = xas.average_polarised_scans(*spectras)
print(pol1)
print(pol2)

if pol2 is None:
    raise  ValueError(f"No opposite polarisations found: {[s.metadata.pol for s in spectras]}")

# Plot averaged scans
fig, axes = plt.subplots(1, n_spectra, figsize=[6 * n_spectra, 6], dpi=100, squeeze=False)
for scan in [pol1, pol2]:
    for ax, (mode, spectra) in zip(axes.flat, scan.spectra.items()):
        spectra.plot(ax)
        ax.set_title(spectra.process_label)
        ax.set_ylabel(mode)
        ax.set_xlabel('E [eV]')
        ax.legend()

### Calculate XMCD / XMLD

In [ ]:
# Calculate XMCD
xmcd = pol1 - pol2
print(xmcd)


In [ ]:
xmcd.create_sum_rules_figure(figsize=(8 * n_spectra, 6), dpi=100);
plt.tight_layout(h_pad=0.1, w_pad=0.1)

# fig, axes = plt.subplots(1, n_spectra, figsize=[6 * n_spectra, 6], dpi=100, squeeze=False)
# for ax, (mode, spectra) in zip(axes.flat, xmcd.spectra.items()):
#     spectra.plot(ax)
#     ax.set_title(spectra.process_label)
#     ax.set_ylabel(mode)
#     ax.set_xlabel('E [eV]')
#     ax.legend()

In [ ]:
# Interactive sum rules calculator
max_signal_ratios = xmcd.calculate_signal_ratio()

def plot_xmcd(mode: str, n_holes: float, split_energy: float):
    spectra = xmcd.spectra[mode]
    orb, spin = spectra.calculate_sum_rules(n_holes, split_energy)

    report = f"Max signal: {max_signal_ratios[mode]:.2%}\n"
    report += f"{xmcd.metadata.element} n_holes = {n_holes}\nL = {orb:.3f} μB\nS = {spin:.3f} μB"
    fig, ax = plt.subplots()
    spectra.plot_sum_rules(ax, split_energy=split_energy)
    ax.text(0.6, 0.1, report, transform=ax.transAxes, fontsize='xx-large')
    ax.set_xlabel('E eV')
    ax.set_title(f"{mode.upper()} {xmcd.name.upper()}")

en = next(iter(xmcd.spectra.values())).energy
widgets.interact(plot_xmcd, mode=list(xmcd.spectra), n_holes=(0.0, 10.0, 0.1), split_energy=(en.min(), en.max(), 1.0))

### Create output NeXus file

In [ ]:
# create processed nexus file
xmcd_filename = f"{spectras[0].metadata.scan_no}-{spectras[-1].metadata.scan_no}_{xmcd.name}.nxs"
xmcd.write_nexus(xmcd_filename)

In [ ]:
# Print structure of file
from mmg_toolbox import data_file_reader

scan = data_file_reader(xmcd_filename)
print(scan.hdf_tree_string())